In [2]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("--- Début du script d'entraînement ---")

# ─────────────────────────────────────────────────────────────
# STEP 1 — Load and prepare data
# ─────────────────────────────────────────────────────────────
df = pd.read_excel('microrna_families_conserved.xlsx') 

df['parasite'] = df['parasite'].str.strip()

# family_name: replace not_found with unknown_family for OHE
df['family_name'] = df['family_name'].fillna('unknown_family')      
df['family_name'] = df['family_name'].replace('not_found', 'unknown_family') 


# scenario: interaction feature parasite × cell type
df['scenario'] = df['parasite'].str.strip() + '_' + df['cell type'].str.strip()

print(f"Total rows         : {len(df)}")
print(f"Target balance     :\n{df['is_upregulated'].value_counts().to_string()}")


# ─────────────────────────────────────────────────────────────
# STEP 2 — Features and target
# ─────────────────────────────────────────────────────────────
categorical_features = [
    'microrna_group_simplified',
    'parasite',
    'organism',
    'cell type',
    'scenario',
    'family_name',      
]

numerical_features = [
    'time',
]

features_to_use = categorical_features + numerical_features

X = df[features_to_use]
y = df['is_upregulated']

print(f"\nCategorical features : {categorical_features}")
print(f"Numerical features   : {numerical_features}")


# ─────────────────────────────────────────────────────────────
# STEP 3 — Build pipeline (OHE + Random Forest + OOB)
# ─────────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        oob_score=True
    ))
])


# ─────────────────────────────────────────────────────────────
# STEP 4 — Train on 100% of data
# ─────────────────────────────────────────────────────────────
print("\nEntraînement du modèle Random Forest sur 100% des données...")
model_pipeline.fit(X, y)
print("Entraînement terminé.")


# ─────────────────────────────────────────────────────────────
# STEP 5 — OOB Score
# ─────────────────────────────────────────────────────────────
oob_accuracy = model_pipeline.named_steps['classifier'].oob_score_
print(f"\nScore OOB : {oob_accuracy:.4f}")
print(f"Le modèle devrait être correct sur environ {oob_accuracy * 100:.2f}% des nouvelles données.")


# ─────────────────────────────────────────────────────────────
# STEP 6 — Feature Importance
# ─────────────────────────────────────────────────────────────
print("\n--- Importance des caractéristiques ---")

preprocessor_fitted = model_pipeline.named_steps['preprocessor']
cat_feature_names = preprocessor_fitted.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + list(cat_feature_names)

importances = model_pipeline.named_steps['classifier'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature':    all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df.head(10).to_string())


# ─────────────────────────────────────────────────────────────
# STEP 7 — Build lookup dicts and save model bundle
# ─────────────────────────────────────────────────────────────
mirna_lookup = (
    df[['microrna', 'microrna_group_simplified', 'family_name', 'mirbase_accession']] 
    .drop_duplicates(subset='microrna')
    .set_index('microrna')
    .to_dict('index')
)

accession_lookup = (
    df[df['mirbase_accession'].notna()]
    [['mirbase_accession', 'microrna', 'microrna_group_simplified', 'family_name']]    
    .drop_duplicates(subset='mirbase_accession')
    .set_index('mirbase_accession')
    .to_dict('index')
)

model_bundle = {
    'model':            model_pipeline,
    'mirna_lookup':     mirna_lookup,
    'accession_lookup': accession_lookup,
    'oob_score':        oob_accuracy,
    'feature_names':    features_to_use,
    'options': {
        'parasite':      sorted(df['parasite'].dropna().unique().tolist()),
        'organism':      sorted(df['organism'].dropna().unique().tolist()),
        'cell_type':     sorted(df['cell type'].dropna().unique().tolist()),
        'time':          sorted(df['time'].dropna().unique().tolist()),
        'seed_families': sorted(df[df['family_name'] != 'unknown_family']['family_name'].unique().tolist()),  
    },
}

model_filename = 'Mir_v2_family_scenario_model.pkl'
joblib.dump(model_bundle, model_filename)
print(f"\nModèle sauvegardé : '{model_filename}'")
print("\n--- Fin du script ---")

--- Début du script d'entraînement ---
Total rows         : 206
Target balance     :
is_upregulated
1    103
0    103

Categorical features : ['microrna_group_simplified', 'parasite', 'organism', 'cell type', 'scenario', 'family_name']
Numerical features   : ['time']

Entraînement du modèle Random Forest sur 100% des données...
Entraînement terminé.

Score OOB : 0.8592
Le modèle devrait être correct sur environ 85.92% des nouvelles données.

--- Importance des caractéristiques ---
                                    Feature  Importance
0                                      time    0.140025
51       microrna_group_simplified_miR-3075    0.031938
29       microrna_group_simplified_miR-1955    0.029503
64       microrna_group_simplified_miR-466p    0.027225
105          family_name_let-7-5p/miR-98-5p    0.025361
143              family_name_unknown_family    0.022858
65       microrna_group_simplified_miR-466q    0.021830
25       microrna_group_simplified_miR-1892    0.020264
101  scena